> **Note**: This notebook calculates environmental severity. It now supports General Waste Detection by using dynamic hazard weights based on the high-level waste group from `configs/classes.yaml`.

# 🌍 Waste Detection — Severity Engine (Notebook 09)

### Overview
This notebook calculates a holistic **Environmental Severity Score (0-100)** for a given image containing waste detections.

### New Features Added
- **Dynamic Hazard Weights**: Replaces the hardcoded 8-class weight dict with a dynamic system based on `waste_group` (plastic=high, paper=low, etc.).
- **Dynamic Color Palette**: Adopts the dynamic HSV colors from NB 08.

### Pipeline Position
```
NB 08 (Inference) → [Predictions] → NB 09 (THIS) → [Severity Reports]
```

## 1. Environment Setup & Imports

In [ ]:
!pip install -q ultralytics rich pyyaml pandas opencv-python matplotlib pillow

In [ ]:
import os
import json
import time
import shutil
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional
from collections import Counter

import yaml
import torch
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from ultralytics import YOLO

console = Console()

## 2. Configuration & Load Model
Load `classes.yaml` for waste group mappings, and load the YOLO model.

In [ ]:
# ==========================================
# Paths & Config
# ==========================================
PROJECT_ROOT = Path('/content/drive/MyDrive/PlasticSense_AI')
MODELS_DIR = PROJECT_ROOT / 'models'
BEST_PT_PATH = MODELS_DIR / 'best.pt'
CONFIGS_DIR = PROJECT_ROOT / 'configs'
CLASSES_YAML = CONFIGS_DIR / 'classes.yaml'

SEVERITY_DIR = PROJECT_ROOT / 'results' / 'severity'
for d in ['json', 'csv', 'visuals']:
    (SEVERITY_DIR / d).mkdir(parents=True, exist_ok=True)

# Load Central Configuration
waste_groups = {}
if CLASSES_YAML.exists():
    with open(CLASSES_YAML, 'r') as f:
        config = yaml.safe_load(f)
    for cat in config.get('categories', []):
        waste_groups[cat['yolo_id']] = cat.get('waste_group', 'unknown')
else:
    console.print("[yellow]⚠ classes.yaml not found.[/yellow]")

# Load Model
console.print(f"[cyan]Loading model from {BEST_PT_PATH}...[/cyan]")
model = YOLO(str(BEST_PT_PATH))
class_names = {int(k): v for k, v in model.names.items()}
console.print(f"[green]✔ Model loaded successfully ({len(class_names)} classes)[/green]")

## 3. Dynamic Hazard Weight Configuration
Instead of hardcoding class names, we map high-level waste groups to hazard weights.

In [ ]:
# ==========================================
# Waste Group Hazard Weights
# ==========================================
# Scale: 1 (least) to 4 (most hazardous)
GROUP_HAZARD_WEIGHTS = {
    'plastic': 3,     # Microplastics, persistent
    'foam': 4,        # Microplastics, difficult to clean (styrofoam)
    'metal': 2,       # Sharp, moderate persistence
    'glass': 2,       # Sharp, very persistent but inert
    'rubber': 3,      # Toxic leaching
    'textile': 2,     # Synthetic fibers
    'paper': 1,       # Biodegradable
    'cardboard': 1,   # Biodegradable
    'wood': 1,        # Biodegradable
    'organic': 1,     # Biodegradable
    'other': 2,       # Batteries, e-waste (could be 4, generalized to 2)
    'unknown': 2
}

MAX_HAZARD_WEIGHT = max(GROUP_HAZARD_WEIGHTS.values())

def get_hazard_weight(cls_id: int) -> int:
    """Get hazard weight for a specific YOLO class ID via its waste group."""
    group = waste_groups.get(cls_id, 'unknown')
    return GROUP_HAZARD_WEIGHTS.get(group, 2)

# Dynamic color generation
np.random.seed(42)
CLASS_COLORS_BGR = {}
for cid in range(len(class_names)):
    hue = int(cid * 180 / max(len(class_names), 1)) % 180
    hsv = np.array([[[hue, 200, 230]]], dtype=np.uint8)
    bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)[0][0]
    CLASS_COLORS_BGR[cid] = (int(bgr[0]), int(bgr[1]), int(bgr[2]))

## 4. Severity Engine Logic

In [ ]:
# ==========================================
# Severity Calculation Functions
# ==========================================

def calculate_density(detections: List[Dict], img_w: int, img_h: int) -> Dict[str, float]:
    img_area = img_w * img_h
    if img_area == 0 or not detections:
        return {'score': 0.0, 'coverage_pct': 0.0}
        
    bbox_area = sum(det['bbox_xywh'][2] * det['bbox_xywh'][3] for det in detections)
    coverage_pct = (bbox_area / img_area) * 100.0
    
    # Cap at 100%, normalize to 60% coverage = max density score of 100
    score = min(100.0, (coverage_pct / 60.0) * 100.0)
    return {'score': score, 'coverage_pct': coverage_pct}

def calculate_hazard(detections: List[Dict]) -> Dict[str, float]:
    if not detections:
        return {'score': 0.0, 'raw': 0, 'max_possible': 0}
        
    raw_score = sum(get_hazard_weight(d['class_id']) for d in detections)
    max_possible = len(detections) * MAX_HAZARD_WEIGHT
    
    score = (raw_score / max_possible) * 100.0 if max_possible > 0 else 0.0
    return {'score': score, 'raw': raw_score, 'max_possible': max_possible}

def check_waterbody(lat=None, lon=None) -> float:
    # Dummy implementation for demo
    # In production, query Google Maps or OSM API
    return 0.0

def calculate_final_severity(detections: List[Dict], img_w: int, img_h: int, lat=None, lon=None) -> Dict:
    """Calculates final severity score (0-100)."""
    count = len(detections)
    
    # 1. Count Score (Maxes out at 50 objects)
    count_score = min(100.0, (count / 50.0) * 100.0)
    
    # 2. Density Score
    density = calculate_density(detections, img_w, img_h)
    
    # 3. Hazard Score
    hazard = calculate_hazard(detections)
    
    # 4. Water Proximity Score
    water_score = check_waterbody(lat, lon)
    
    # Weighted Sum
    # Count: 35%, Density: 30%, Hazard: 25%, Water: 10%
    final_score = (
        (count_score * 0.35) +
        (density['score'] * 0.30) +
        (hazard['score'] * 0.25) +
        (water_score * 0.10)
    )
    
    # Determine level
    level, badge, color = "Low", "🟢", "#2ecc71"
    if final_score > 75:   level, badge, color = "Critical", "🔴", "#e74c3c"
    elif final_score > 50: level, badge, color = "High", "🟠", "#e67e22"
    elif final_score > 25: level, badge, color = "Medium", "🟡", "#f1c40f"
    
    return {
        'final_score': round(final_score, 1),
        'level': level,
        'badge': badge,
        'color': color,
        'components': {
            'count_score': round(count_score, 1),
            'density_score': round(density['score'], 1),
            'hazard_score': round(hazard['score'], 1),
            'water_score': round(water_score, 1)
        },
        'stats': {
            'object_count': count,
            'coverage_pct': round(density['coverage_pct'], 1),
            'raw_hazard': hazard['raw']
        }
    }

## 5. End-to-End Analysis
Run YOLO inference, compute severity, and visualize.

In [ ]:
def analyze_image(img_path: Path):
    # 1. Predict
    img = cv2.imread(str(img_path))
    if img is None: return
    img_h, img_w = img.shape[:2]
    
    results = model.predict(source=img, conf=0.25, verbose=False)[0]
    
    detections = []
    if results.boxes is not None:
        for box in results.boxes:
            cid = int(box.cls.item())
            xyxy = box.xyxy[0].cpu().numpy().tolist()
            xywh = box.xywh[0].cpu().numpy().tolist()
            detections.append({
                'class_id': cid,
                'class': class_names.get(cid, str(cid)),
                'bbox_xyxy': xyxy,
                'bbox_xywh': xywh
            })
            
    # 2. Calculate Severity
    severity = calculate_final_severity(detections, img_w, img_h)
    
    # 3. Output
    console.print(Panel.fit(
        f"[bold]Image:[/bold] {img_path.name}\n"
        f"[bold]Objects:[/bold] {severity['stats']['object_count']}\n"
        f"[bold]Coverage:[/bold] {severity['stats']['coverage_pct']}%\n"
        f"[bold]Severity Score:[/bold] {severity['final_score']} / 100 {severity['badge']} ([bold]{severity['level']}[/bold])"
    ))
    return severity

# Test on one image
TEST_IMAGES = PROJECT_ROOT / 'datasets' / 'taco_yolo_augmented' / 'images' / 'test'
test_imgs = list(TEST_IMAGES.glob('*.[jJ][pP][gG]'))
if test_imgs:
    analyze_image(test_imgs[0])